# 04 – Data Leakage Check
**Szerző:** Magda Ferenc (U5O0BB)  
**Projekt:** Gitár akkord-felismerő szoftver gépi látással  
**Notebook célja:** Megbizonyosodni arról, hogy az `all/` mappából készített train/val/test  
split *nem* tartalmaz szivárgást (data leakage), azaz:

1. **Fájlnév-duplikáció** – ugyanaz a kép nem szerepel két splitben sem.  
2. **Képtartalom-duplikáció (perceptual hash)** – vizuálisan azonos képek nem kerültek  
   különböző splitek közé (pl. augmentált másolatok, átnevezett fájlok).  
3. **Osztályazonosság** – minden kép pontosan egy osztályban szerepel.  
4. **Pixelszintű duplikáció** – md5/sha256 hash alapú egyezés.  
5. **Összefoglaló audit-táblázat** – az összes lelet áttekintése.

---
**Stratégia:** manifest-alapú ellenőrzés – a `data/split_manifest.csv` egyetlen  
igazságforrás; az ellenőrzések mind ebből indulnak ki.


## 1. Könyvtárak és konfiguráció

In [1]:
import hashlib
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

# imagehash a perceptual hash-hez (pip install imagehash)
try:
    import imagehash
    IMAGEHASH_AVAILABLE = True
except ImportError:
    IMAGEHASH_AVAILABLE = False
    print("⚠️  imagehash csomag nem elérhető – "
          "telepítsd: pip install imagehash\n"
          "   A perceptual-hash ellenőrzés ki lesz hagyva.")

warnings.filterwarnings("ignore")

NOTEBOOK_DIR  = Path(__file__).parent if "__file__" in dir() else Path.cwd()
PROJECT_ROOT  = NOTEBOOK_DIR.parent
DATA_ROOT     = PROJECT_ROOT / "data"
MANIFEST_PATH = DATA_ROOT / "split_manifest.csv"
OUTPUT_DIR    = PROJECT_ROOT / "output" / "04_data_leakage_check"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST_PATH.exists(), f"Manifest nem található: {MANIFEST_PATH}"
print(f"✅  Manifest: {MANIFEST_PATH}")
print(f"📁  Output:   {OUTPUT_DIR}")


✅  Manifest: /data/Tanulmányok/Képfeldolgozás alap algoritmusai/Féléves feladat/guitar-chord-recognition/data/split_manifest.csv
📁  Output:   /data/Tanulmányok/Képfeldolgozás alap algoritmusai/Féléves feladat/guitar-chord-recognition/output/04_data_leakage_check


## 2. Manifest betöltése

In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Sorok száma: {len(manifest)}")
print(f"Oszlopok:    {list(manifest.columns)}")
print()
print(manifest.groupby("split").size().rename("db"))


Sorok száma: 297
Oszlopok:    ['split', 'class', 'filename', 'path', 'size_kb']

split
test      45
train    207
val       45
Name: db, dtype: int64


## 3. Fájlnév-duplikáció (split-szintű)

Ugyanaz a `filename` nem jelenhet meg több splitben.  
A `(class, filename)` pár alapján is ellenőrzünk.


In [3]:
leakage_filename = {}

# Globális filename-duplikáció
dup_fname = manifest[manifest.duplicated(subset=["filename"], keep=False)]
if dup_fname.empty:
    print("✅  Fájlnév-duplikáció: NINCS")
else:
    print(f"❌  Fájlnév-duplikáció: {len(dup_fname)} sor érintett!")
    print(dup_fname[["split","class","filename"]].sort_values("filename").to_string(index=False))
    leakage_filename["fájlnév-dup"] = dup_fname

# Cross-split szivárgás: filename azonos, split különböző
cross = (manifest.groupby("filename")["split"]
         .nunique()
         .reset_index()
         .rename(columns={"split": "n_splits"}))
cross_leak = cross[cross["n_splits"] > 1]
if cross_leak.empty:
    print("✅  Cross-split fájlnév-szivárgás: NINCS")
else:
    print(f"❌  Cross-split szivárgás: {len(cross_leak)} fájl!")
    print(cross_leak.to_string(index=False))
    leakage_filename["cross_split"] = cross_leak


✅  Fájlnév-duplikáció: NINCS
✅  Cross-split fájlnév-szivárgás: NINCS


## 4. Pixelszintű duplikáció (MD5 hash)

Bájt-azonos fájlok nem kerülhetnek különböző splitek közé.


In [4]:
def md5_of_file(path: str) -> str:
    h = hashlib.md5()
    try:
        with open(path, "rb") as f:
            h.update(f.read())
    except FileNotFoundError:
        return "FILE_NOT_FOUND"
    return h.hexdigest()

print("MD5 hash kiszámítása... (ez eltarthat egy percig)")
manifest["md5"] = manifest["path"].apply(md5_of_file)

missing = (manifest["md5"] == "FILE_NOT_FOUND").sum()
if missing:
    print(f"⚠️  {missing} fájl nem található a lemezen "
          "(a manifest ellenőrzéséhez futtasd a 02-es notebookot).")

# Cross-split MD5-szivárgás
md5_splits = (manifest.groupby("md5")["split"]
              .nunique()
              .reset_index()
              .rename(columns={"split": "n_splits"}))
md5_leak = md5_splits[md5_splits["n_splits"] > 1]

if md5_leak.empty:
    print("✅  MD5-alapú cross-split duplikáció: NINCS")
else:
    detail = manifest[manifest["md5"].isin(md5_leak["md5"])].sort_values(["md5","split"])
    print(f"❌  MD5-szivárgás: {len(detail)} kép ({len(md5_leak)} egyedi hash)")
    print(detail[["split","class","filename","md5"]].to_string(index=False))

# Duplikáció azonos spliten belül (informális)
same_split_dup = (manifest.groupby(["split","md5"])["filename"]
                  .count()
                  .reset_index()
                  .rename(columns={"filename":"count"}))
intra_dup = same_split_dup[same_split_dup["count"] > 1]
if intra_dup.empty:
    print("✅  Azonos spliten belüli MD5-duplikáció: NINCS")
else:
    print(f"ℹ️  Azonos spliten belüli duplikáció: {intra_dup['count'].sum()} sor (nem szivárgás, de érdemes tudni)")
    print(intra_dup.to_string(index=False))


MD5 hash kiszámítása... (ez eltarthat egy percig)
✅  MD5-alapú cross-split duplikáció: NINCS
✅  Azonos spliten belüli MD5-duplikáció: NINCS


## 5. Perceptual hash – vizuálisan hasonló képek (pHash, threshold ≤ 4 bit)

A perceptual hash (pHash) kis küszöbértékkel (≤ 4 Hamming-távolság) megtalálja  
az augmentáció által létrehozott „majdnem azonos" képeket is.  
Ez az erősebb szivárgás-ellenőrzés.

> **Ajánlás:** threshold = 4 (8×8 DCT-hash esetén konzervatív, de megbízható).


In [5]:
PHASH_THRESHOLD = 4   # Hamming-távolság küszöb

if not IMAGEHASH_AVAILABLE:
    print("⚠️  imagehash nem elérhető – ez a cella ki van hagyva.")
    print("   Telepítés: pip install imagehash")
else:
    print("pHash kiszámítása...")
    def phash_of_file(path: str):
        try:
            return imagehash.phash(Image.open(path).convert("RGB"))
        except Exception:
            return None

    manifest["phash"] = manifest["path"].apply(phash_of_file)
    none_cnt = manifest["phash"].isna().sum()
    if none_cnt:
        print(f"⚠️  {none_cnt} fájlhoz nem sikerült pHash-t számítani.")

    # Cross-split hasonlóság-ellenőrzés
    train_rows = manifest[manifest["split"] == "train"].dropna(subset=["phash"])
    val_rows   = manifest[manifest["split"] == "val"].dropna(subset=["phash"])
    test_rows  = manifest[manifest["split"] == "test"].dropna(subset=["phash"])

    leaks = []
    for ref_split, ref_df in [("val", val_rows), ("test", test_rows)]:
        for _, row in ref_df.iterrows():
            for _, train_row in train_rows.iterrows():
                dist = row["phash"] - train_row["phash"]
                if dist <= PHASH_THRESHOLD:
                    leaks.append({
                        "ref_split":   ref_split,
                        "ref_file":    row["filename"],
                        "ref_class":   row["class"],
                        "train_file":  train_row["filename"],
                        "train_class": train_row["class"],
                        "hamming":     dist,
                    })

    if not leaks:
        print(f"✅  Perceptual hash (threshold={PHASH_THRESHOLD}): "
              "cross-split vizuális duplikáció NINCS")
    else:
        df_leaks = pd.DataFrame(leaks).sort_values("hamming")
        print(f"❌  {len(df_leaks)} potenciális vizuális szivárgás (threshold={PHASH_THRESHOLD})!")
        print(df_leaks.to_string(index=False))
        df_leaks.to_csv(OUTPUT_DIR / "phash_leaks.csv", index=False)
        print(f"   → Mentve: {OUTPUT_DIR}/phash_leaks.csv")


pHash kiszámítása...
❌  138 potenciális vizuális szivárgás (threshold=4)!
ref_split                  ref_file ref_class                train_file train_class  hamming
      val   IMG_20251102_023916.jpg         F   IMG_20251102_023917.jpg           F        0
      val   IMG_20251102_023916.jpg         F IMG_20251102_023918_1.jpg           F        0
      val   IMG_20251102_023836.jpg   No hand   IMG_20251102_023835.jpg     No hand        0
      val   IMG_20251102_023836.jpg   No hand IMG_20251102_023834_1.jpg     No hand        0
     test   IMG_20251102_024042.jpg         C   IMG_20251102_024041.jpg           C        0
     test IMG_20251102_024136_1.jpg         B   IMG_20251102_024136.jpg           B        0
     test         1762212326478.jpg         A         1762212326513.jpg           A        0
     test         1762212326478.jpg         A         1762212326435.jpg           A        0
     test   IMG_20251102_024035.jpg         C   IMG_20251102_024036.jpg           C      

## 6. Osztályazonosság ellenőrzése

Minden fájlnévnek pontosan egy osztálya lehet a manifestben.


In [6]:
class_per_file = (manifest.groupby("filename")["class"]
                  .nunique()
                  .reset_index()
                  .rename(columns={"class": "n_classes"}))
multi_class = class_per_file[class_per_file["n_classes"] > 1]

if multi_class.empty:
    print("✅  Osztályazonosság: minden fájlnak pontosan 1 osztálya van.")
else:
    print(f"❌  {len(multi_class)} fájlhoz több osztály is tartozik!")
    print(multi_class.to_string(index=False))


✅  Osztályazonosság: minden fájlnak pontosan 1 osztálya van.


## 7. Összefoglaló audit-táblázat

In [7]:
import datetime

checks = [
    ("Fájlnév-duplikáció (cross-split)",   "cross_split szivárgás?",
     "dup_fname.empty and cross_leak.empty"),
    ("MD5-alapú szivárgás (cross-split)",   "bájt-azonos képek?",
     "md5_leak.empty"),
    ("MD5-duplikáció (intra-split)",         "azonos spliten belül?",
     "intra_dup.empty"),
    ("Osztályazonosság",                     "1 fájl = 1 osztály?",
     "multi_class.empty"),
]

rows = []
for name, question, expr in checks:
    try:
        result = eval(expr)
        status = "✅ OK" if result else "❌ PROBLÉMA"
    except Exception as e:
        status = f"⚠️ Hiba: {e}"
    rows.append({"Ellenőrzés": name, "Kérdés": question, "Eredmény": status})

# pHash sor hozzáadása
if IMAGEHASH_AVAILABLE:
    p_ok = len(leaks) == 0
    rows.append({"Ellenőrzés": f"Perceptual hash (threshold={PHASH_THRESHOLD})",
                 "Kérdés": "vizuálisan hasonló képek cross-split?",
                 "Eredmény": "✅ OK" if p_ok else "❌ PROBLÉMA"})
else:
    rows.append({"Ellenőrzés": "Perceptual hash",
                 "Kérdés": "vizuálisan hasonló képek cross-split?",
                 "Eredmény": "⚠️ imagehash nem telepítve"})

audit_df = pd.DataFrame(rows)
print("=" * 65)
print("  DATA LEAKAGE AUDIT ÖSSZEFOGLALÓ")
print(f"  {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 65)
print(audit_df.to_string(index=False))
print("=" * 65)

audit_df.to_csv(OUTPUT_DIR / "leakage_audit.csv", index=False)
print(f"\nAudit CSV mentve: {OUTPUT_DIR}/leakage_audit.csv")


  DATA LEAKAGE AUDIT ÖSSZEFOGLALÓ
  2026-05-13 02:48
                       Ellenőrzés                                Kérdés   Eredmény
 Fájlnév-duplikáció (cross-split)                cross_split szivárgás?       ✅ OK
MD5-alapú szivárgás (cross-split)                    bájt-azonos képek?       ✅ OK
     MD5-duplikáció (intra-split)                 azonos spliten belül?       ✅ OK
                 Osztályazonosság                   1 fájl = 1 osztály?       ✅ OK
    Perceptual hash (threshold=4) vizuálisan hasonló képek cross-split? ❌ PROBLÉMA

Audit CSV mentve: /data/Tanulmányok/Képfeldolgozás alap algoritmusai/Féléves feladat/guitar-chord-recognition/output/04_data_leakage_check/leakage_audit.csv


## 8. Összefoglaló és következő lépések

### Mit értünk el?
- Megvizsgáltuk a split-manifestet **4 dimenzióban** (fájlnév, MD5, pHash, osztály).
- Az audit-táblázat CSV-ben is mentésre kerül (`output/04_data_leakage_check/leakage_audit.csv`).

### Ha probléma van
| Lelet | Teendő |
|---|---|
| Fájlnév cross-split szivárgás | Futtasd újra a `02_split_manifest.ipynb`-t, ellenőrizd a split-logikát |
| MD5-szivárgás | Távolítsd el a duplikátumokat az `all/` mappából, majd re-split |
| pHash-szivárgás | Döntés: elfogadjuk (kis adathalmaz) vagy kizárjuk a hasonló párokat |
| Több osztály / fájl | Ellenőrizd az `all/` mappa mappaszerkezetét |

### Következő lépés
→ `05_model_baseline.ipynb` – alap CNN / transfer learning modell felépítése
